# Modulo 4 · Lezione 3 — Query avanzate: JOIN, GROUP BY e subquery

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Argomento | A cosa serve |
|---|------|----------------|
| 1 | `INNER JOIN` | Combinare righe che hanno corrispondenza in entrambe le tabelle |
| 2 | `LEFT JOIN` | Includere tutte le righe della tabella sinistra |
| 3 | `RIGHT JOIN` / `FULL JOIN` | Varianti di JOIN esterno |
| 4 | `GROUP BY` | Raggruppare righe e aggregare per gruppo |
| 5 | `HAVING` | Filtrare i gruppi (come WHERE ma per i gruppi) |
| 6 | Subquery | Annidare una SELECT dentro un'altra |
| 7 | Pratica | Analisi delle vendite del negozio |

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")

def q(sql, *params):
    return pd.read_sql_query(sql, conn, params=params)

# Sguardo rapido alle tabelle disponibili
print("Tabelle:", q("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist())

---
## 1. Il concetto di JOIN

Un `JOIN` combina colonne di due (o più) tabelle collegate tramite una condizione.

```
clientes                          pedidos
┌────┬──────────┬────────┐        ┌─────┬────────────┬───────────┐
│ id │ nombre   │ ciudad │        │ id  │ cliente_id │ estado    │
├────┼──────────┼────────┤        ├─────┼────────────┼───────────┤
│  1 │ Ana      │ Madrid │◄──────►│ 101 │     1      │ entregado │
│  2 │ Carlos   │ BCN    │◄──────►│ 103 │     2      │ entregado │
│  3 │ Lucía    │ Sevilla│        │ 104 │     3      │ entregado │
└────┴──────────┴────────┘        └─────┴────────────┴───────────┘

JOIN ON clientes.id = pedidos.cliente_id

┌────────────┬──────────────────────────────┐
│ c.nombre   │ p.id   p.estado              │
├────────────┼──────────────────────────────┤
│ Ana        │ 101    entregado             │
│ Carlos     │ 103    entregado             │
│ Lucía      │ 104    entregado             │
└────────────┴──────────────────────────────┘
```

**Alias di tabella**: quando usi un JOIN è comune dare alias brevi alle tabelle per non ripeterne il nome completo:
```sql
FROM clientes c
JOIN pedidos p ON c.id = p.cliente_id
```

---
## 1. `INNER JOIN`

Restituisce solo le righe che hanno corrispondenza **in entrambe le tabelle**. Se una riga della tabella A non ha nessuna corrispondente in B, non compare nel risultato.

```sql
SELECT col1, col2
FROM   tabla_a a
INNER JOIN tabla_b b ON a.fk = b.id;

-- INNER è la parola chiave predefinita; queste due sono equivalenti:
FROM tabla_a JOIN tabla_b ON ...
FROM tabla_a INNER JOIN tabla_b ON ...
```

In [ ]:
# Ordini con il nome del cliente
print("=== Ordini con nome del cliente ===")
display(q("""
    SELECT p.id          AS pedido_id,
           c.nombre      AS cliente,
           c.ciudad,
           p.fecha,
           p.estado
    FROM   pedidos   p
    JOIN   clientes  c ON p.cliente_id = c.id
    ORDER  BY p.fecha
"""))

# Prodotti con nome della categoria
print("\n=== Prodotti con categoria ===")
display(q("""
    SELECT prod.nombre   AS producto,
           cat.nombre    AS categoria,
           prod.precio,
           prod.stock
    FROM   productos  prod
    JOIN   categorias cat ON prod.categoria_id = cat.id
    ORDER  BY cat.nombre, prod.precio DESC
"""))

# JOIN di tre tabelle: righe d'ordine con tutti i dettagli
print("\n=== Dettaglio completo degli ordini ===")
display(q("""
    SELECT p.id           AS pedido,
           c.nombre       AS cliente,
           prod.nombre    AS producto,
           pp.cantidad,
           pp.precio_ud,
           ROUND(pp.cantidad * pp.precio_ud, 2) AS subtotal
    FROM   pedido_productos pp
    JOIN   pedidos   p    ON pp.pedido_id   = p.id
    JOIN   clientes  c    ON p.cliente_id   = c.id
    JOIN   productos prod ON pp.producto_id = prod.id
    ORDER  BY p.id, prod.nombre
"""))

---
## 2. `LEFT JOIN`

Restituisce **tutte le righe della tabella sinistra** (quella nella `FROM`), e le colonne della tabella destra riempite con `NULL` quando non c'è corrispondenza.

```
clientes (A)    pedidos (B)        Risultato LEFT JOIN
┌────┬──────┐   ┌─────┬──────┐    ┌──────┬─────────┐
│  1 │ Ana  │   │ 101 │  1   │    │ Ana  │ 101     │
│  2 │ Bob  │   │ 102 │  1   │    │ Ana  │ 102     │
│  3 │ Cara │   │ 103 │  2   │    │ Bob  │ 103     │
└────┴──────┘   └─────┴──────┘    │ Cara │ NULL    │ ← senza ordini
                                   └──────┴─────────┘
```

Caso d'uso principale: **trovare righe senza corrispondenza** (`WHERE b.id IS NULL`).

In [ ]:
# Clienti E il loro numero di ordini (compresi quelli che non hanno mai ordinato)
print("=== Clienti con numero di ordini (LEFT JOIN) ===")
display(q("""
    SELECT c.nombre,
           c.ciudad,
           COUNT(p.id) AS num_pedidos
    FROM   clientes c
    LEFT JOIN pedidos p ON c.id = p.cliente_id
    GROUP  BY c.id, c.nombre, c.ciudad
    ORDER  BY num_pedidos DESC
"""))

# Clienti che NON hanno mai ordinato
print("\n=== Clienti senza alcun ordine ===")
display(q("""
    SELECT c.nombre, c.email
    FROM   clientes c
    LEFT JOIN pedidos p ON c.id = p.cliente_id
    WHERE  p.id IS NULL
"""))

# Prodotti mai ordinati
print("\n=== Prodotti mai venduti ===")
display(q("""
    SELECT prod.nombre, prod.precio, prod.stock
    FROM   productos prod
    LEFT JOIN pedido_productos pp ON prod.id = pp.producto_id
    WHERE  pp.producto_id IS NULL
"""))

---
## 3. `GROUP BY` — Raggruppare e aggregare

```sql
SELECT   col_agrupacion, AGG(col)
FROM     tabla
[JOIN ...]
[WHERE  condición_de_fila]
GROUP BY col_agrupacion
[HAVING condición_de_grupo]
ORDER BY ...
```

**Regola chiave**: in una `SELECT` con `GROUP BY`, ogni colonna nella `SELECT` deve essere:
- Una delle colonne di raggruppamento, **oppure**
- L'argomento di una funzione di aggregazione.

```
PRIMA del GROUP BY — righe individuali:
ciudad      precio
Madrid      599.99
Madrid       45.00
Barcelona   350.00

DOPO il GROUP BY ciudad:
ciudad      COUNT(*)   AVG(precio)
Madrid        2          322.50
Barcelona     1          350.00
```

In [ ]:
# Numero di prodotti per categoria
print("=== Prodotti per categoria ===")
display(q("""
    SELECT cat.nombre              AS categoria,
           COUNT(prod.id)          AS num_productos,
           ROUND(AVG(prod.precio), 2) AS precio_medio,
           SUM(prod.stock)         AS stock_total
    FROM   categorias cat
    JOIN   productos  prod ON prod.categoria_id = cat.id
    GROUP  BY cat.id, cat.nombre
    ORDER  BY num_productos DESC
"""))

# Fatturato per ordine (totale di ogni ordine)
print("\n=== Totale per ordine ===")
display(q("""
    SELECT p.id              AS pedido_id,
           c.nombre          AS cliente,
           p.fecha,
           p.estado,
           COUNT(pp.producto_id)                      AS num_lineas,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)  AS total
    FROM   pedidos          p
    JOIN   clientes         c  ON p.cliente_id   = c.id
    JOIN   pedido_productos pp ON p.id           = pp.pedido_id
    GROUP  BY p.id, c.nombre, p.fecha, p.estado
    ORDER  BY total DESC
"""))

# Ordini e spesa per cliente
print("\n=== Spesa totale per cliente ===")
display(q("""
    SELECT c.nombre                                    AS cliente,
           COUNT(DISTINCT p.id)                        AS pedidos,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)   AS gasto_total
    FROM   clientes         c
    JOIN   pedidos          p  ON c.id          = p.cliente_id
    JOIN   pedido_productos pp ON p.id          = pp.pedido_id
    GROUP  BY c.id, c.nombre
    ORDER  BY gasto_total DESC
"""))

---
## 4. `HAVING` — Filtrare i gruppi

```
WHERE  → filtra le righe PRIMA del raggruppamento
HAVING → filtra i gruppi DOPO il raggruppamento
```

```sql
SELECT categoria_id, AVG(precio) AS media
FROM productos
WHERE stock > 0           -- elimina i prodotti senza stock prima di raggruppare
GROUP BY categoria_id
HAVING AVG(precio) > 100  -- solo categorie con prezzo medio > 100
```

In [ ]:
# Clienti con più di un ordine
print("=== Clienti con più di 1 ordine ===")
display(q("""
    SELECT c.nombre, COUNT(p.id) AS pedidos
    FROM   clientes c
    JOIN   pedidos  p ON c.id = p.cliente_id
    GROUP  BY c.id, c.nombre
    HAVING COUNT(p.id) > 1
    ORDER  BY pedidos DESC
"""))

# Categorie il cui prezzo medio supera 200€
print("\n=== Categorie con prezzo medio > 200€ ===")
display(q("""
    SELECT cat.nombre                  AS categoria,
           ROUND(AVG(prod.precio), 2)  AS precio_medio
    FROM   categorias cat
    JOIN   productos  prod ON prod.categoria_id = cat.id
    GROUP  BY cat.id, cat.nombre
    HAVING AVG(prod.precio) > 200
    ORDER  BY precio_medio DESC
"""))

# Prodotti più venduti (più di 1 unità in totale)
print("\n=== Prodotti venduti per >1 unità totale ===")
display(q("""
    SELECT prod.nombre,
           SUM(pp.cantidad) AS unidades_vendidas
    FROM   productos        prod
    JOIN   pedido_productos pp ON prod.id = pp.producto_id
    GROUP  BY prod.id, prod.nombre
    HAVING SUM(pp.cantidad) > 1
    ORDER  BY unidades_vendidas DESC
"""))

---
## 5. Subquery

Una subquery è una `SELECT` annidata dentro un'altra `SELECT`.

```sql
-- In WHERE: confrontare con un valore calcolato
SELECT nombre, precio
FROM productos
WHERE precio > (SELECT AVG(precio) FROM productos);

-- In WHERE con IN: filtrare per un elenco
SELECT nombre FROM clientes
WHERE id IN (SELECT cliente_id FROM pedidos WHERE estado = 'pendiente');

-- In FROM: trattare il risultato come tabella temporanea
SELECT cat, media
FROM (SELECT categoria_id AS cat, AVG(precio) AS media
      FROM productos GROUP BY categoria_id) sub
WHERE media > 100;
```

In [ ]:
# Prodotti sopra il prezzo medio
print("=== Prodotti più cari della media ===")
display(q("""
    SELECT nombre,
           precio,
           ROUND((SELECT AVG(precio) FROM productos), 2) AS precio_medio
    FROM productos
    WHERE precio > (SELECT AVG(precio) FROM productos)
    ORDER BY precio DESC
"""))

# Clienti con ordini in attesa
print("\n=== Clienti con ordini in attesa ===")
display(q("""
    SELECT nombre, email
    FROM   clientes
    WHERE  id IN (
        SELECT DISTINCT cliente_id
        FROM   pedidos
        WHERE  estado = 'pendiente'
    )
"""))

# L'ordine più costoso di ogni cliente
print("\n=== Ordine di importo maggiore per cliente ===")
display(q("""
    SELECT c.nombre AS cliente,
           sub.pedido_id,
           sub.total
    FROM (
        SELECT p.cliente_id,
               p.id AS pedido_id,
               ROUND(SUM(pp.cantidad * pp.precio_ud), 2) AS total
        FROM   pedidos          p
        JOIN   pedido_productos pp ON p.id = pp.pedido_id
        GROUP  BY p.id, p.cliente_id
    ) sub
    JOIN clientes c ON sub.cliente_id = c.id
    WHERE sub.total = (
        SELECT MAX(sub2.total)
        FROM (
            SELECT p2.cliente_id,
                   SUM(pp2.cantidad * pp2.precio_ud) AS total
            FROM   pedidos          p2
            JOIN   pedido_productos pp2 ON p2.id = pp2.pedido_id
            GROUP  BY p2.id, p2.cliente_id
        ) sub2
        WHERE sub2.cliente_id = sub.cliente_id
    )
    ORDER BY sub.total DESC
"""))

---
## 6. Pratica — Analisi delle vendite

---

### Esercizio 1 — JOIN di base

**Consegna:** Scrivi una query che restituisca:
- Nome del prodotto, nome della categoria, prezzo e stock.
- Solo prodotti della categoria «Informática».
- Ordinati per prezzo decrescente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
display(q("""
    SELECT prod.nombre      AS producto,
           cat.nombre       AS categoria,
           prod.precio,
           prod.stock
    FROM   productos  prod
    JOIN   categorias cat ON prod.categoria_id = cat.id
    WHERE  cat.nombre = 'Informática'
    ORDER  BY prod.precio DESC
"""))
```
</details>

---
### Esercizio 2 — GROUP BY

**Consegna:** Calcola, per stato dell'ordine (`pendiente`, `enviado`, `entregado`):
- Numero di ordini.
- Importo totale accumulato.
- Importo medio per ordine.

Ordina per importo totale decrescente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
display(q("""
    SELECT p.estado,
           COUNT(DISTINCT p.id)                        AS num_pedidos,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)   AS total_acumulado,
           ROUND(AVG(pp.cantidad * pp.precio_ud), 2)   AS media_linea
    FROM   pedidos          p
    JOIN   pedido_productos pp ON p.id = pp.pedido_id
    GROUP  BY p.estado
    ORDER  BY total_acumulado DESC
"""))
```
</details>

---
### Esercizio 3 — HAVING

**Consegna:** Trova i prodotti che hanno generato più di 200€ in vendite totali. Mostra nome, categoria, unità vendute e importo totale. Ordina per importo decrescente.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
display(q("""
    SELECT prod.nombre                                 AS producto,
           cat.nombre                                  AS categoria,
           SUM(pp.cantidad)                            AS unidades_vendidas,
           ROUND(SUM(pp.cantidad * pp.precio_ud), 2)   AS total_ventas
    FROM   productos        prod
    JOIN   categorias       cat ON prod.categoria_id = cat.id
    JOIN   pedido_productos pp  ON prod.id           = pp.producto_id
    GROUP  BY prod.id, prod.nombre, cat.nombre
    HAVING SUM(pp.cantidad * pp.precio_ud) > 200
    ORDER  BY total_ventas DESC
"""))
```
</details>

---
### Esercizio 4 — LEFT JOIN per individuare lacune

**Consegna:**
1. Elenca le categorie che non hanno alcun prodotto.
2. Elenca i clienti che hanno ordini ma tutti in stato «pendiente» (non hanno mai ricevuto un ordine).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
# 1. Categorie senza prodotti
display(q("""
    SELECT cat.nombre AS categoria
    FROM   categorias cat
    LEFT JOIN productos prod ON prod.categoria_id = cat.id
    WHERE  prod.id IS NULL
"""))

# 2. Clienti i cui ordini sono tutti 'pendiente'
display(q("""
    SELECT c.nombre, COUNT(p.id) AS pedidos_pendientes
    FROM   clientes c
    JOIN   pedidos  p ON c.id = p.cliente_id
    GROUP  BY c.id, c.nombre
    HAVING COUNT(p.id) = SUM(p.estado = 'pendiente')
"""))
```
</details>

---
### Esercizio 5 (sfida) — Funzione `ranking_ventas`

**Consegna:** Scrivi la funzione `ranking_ventas(conn, top_n=5)` che restituisca un DataFrame con la classifica dei `top_n` prodotti più venduti (per importo totale), includendo:
- Posizione in classifica.
- Nome del prodotto e categoria.
- Unità vendute, importo totale e prezzo medio di vendita.
- Quanti ordini distinti includevano quel prodotto.
- Se il prodotto ha stock sufficiente a coprire la domanda media (stock >= unidades_vendidas / numero_pedidos).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import pandas as pd

def ranking_ventas(conn, top_n=5):
    df = pd.read_sql_query(f"""
        SELECT prod.nombre                                  AS producto,
               cat.nombre                                   AS categoria,
               SUM(pp.cantidad)                             AS unidades,
               ROUND(SUM(pp.cantidad * pp.precio_ud), 2)    AS importe_total,
               ROUND(AVG(pp.precio_ud), 2)                  AS precio_medio_venta,
               COUNT(DISTINCT pp.pedido_id)                 AS num_pedidos,
               prod.stock
        FROM   productos        prod
        JOIN   categorias       cat ON prod.categoria_id = cat.id
        JOIN   pedido_productos pp  ON prod.id           = pp.producto_id
        GROUP  BY prod.id, prod.nombre, cat.nombre, prod.stock
        ORDER  BY importe_total DESC
        LIMIT  {top_n}
    """, conn)

    df.insert(0, 'ranking', range(1, len(df) + 1))
    df['demanda_media'] = (df['unidades'] / df['num_pedidos']).round(1)
    df['stock_ok'] = df['stock'] >= df['demanda_media']
    df['stock_ok'] = df['stock_ok'].map({True: '✅', False: '⚠️'})
    return df.drop(columns=['stock'])

display(ranking_ventas(conn, top_n=5))
```
</details>

---
## Riepilogo della lezione

```sql
-- Tipi di JOIN
FROM a INNER JOIN b ON a.id = b.fk    -- solo corrispondenze
FROM a LEFT  JOIN b ON a.id = b.fk    -- tutte le righe di a

-- Ordine di valutazione
FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT

-- GROUP BY + HAVING
SELECT col, COUNT(*), SUM(x)
FROM tabla
WHERE condicion_de_fila          -- filtra PRIMA di raggruppare
GROUP BY col
HAVING COUNT(*) > 3              -- filtra DOPO aver raggruppato
ORDER BY SUM(x) DESC;

-- Subquery scalare
WHERE precio > (SELECT AVG(precio) FROM productos)

-- Subquery con IN
WHERE id IN (SELECT cliente_id FROM pedidos WHERE estado = 'pendiente')
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 4** impareremo a **progettare** database: normalizzazione (1FN, 2FN, 3FN) e tipi di relazioni (1:1, 1:N, N:M).

In [ ]:
conn.close()
print("Connessione chiusa")